In [2]:
from reportlab.platypus import PageBreak
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

DATA_FILE = "../data/retail_sales_dataset.csv"
PDF_FILE = "Retail_Sales_Pie_Chart_Report.pdf"
PLOT_DIR = "plots"

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

df = pd.read_csv(DATA_FILE)

sales_col = None
for c in df.columns:
    if any(k in c.lower() for k in ["total", "amount", "revenue", "sales"]):
        sales_col = c
        break
if not sales_col:
    if "Quantity" in df.columns and "Price per Unit" in df.columns:
        df["Total Amount"] = df["Quantity"] * df["Price per Unit"]
        sales_col = "Total Amount"
    else:
        raise Exception("No sales column found!")

df[sales_col] = pd.to_numeric(df[sales_col], errors="coerce")
df.dropna(subset=[sales_col], inplace=True)

if "Age" in df.columns:
    bins = [0, 17, 25, 35, 45, 60, 100]
    labels = ['<=17', '18-25', '26-35', '36-45', '46-60', '60+']
    df['AgeGroup'] = pd.cut(df["Age"], bins=bins, labels=labels, right=True)

product_col = next((c for c in df.columns if "product" in c.lower()), None)
gender_col = next((c for c in df.columns if "gender" in c.lower()), None)
cust_col = next((c for c in df.columns if "customer" in c.lower() and "id" not in c.lower()), None)
id_col = next((c for c in df.columns if "customer id" in c.lower()), None)

def plot_pie(series, title, filename):
    s = series.sort_values(ascending=False)
    total = s.sum()
    s = s[s > 0]
    other_threshold = 0.03
    small = s[s / total < other_threshold]
    if len(small) > 0:
        s = s[s / total >= other_threshold]
        s["Other"] = small.sum()
    plt.figure(figsize=(5, 5))
    plt.pie(s, labels=s.index, autopct="%1.1f%%", startangle=90)
    plt.title(title)
    plt.tight_layout()
    filepath = os.path.join(PLOT_DIR, filename)
    plt.savefig(filepath, dpi=200)
    plt.close()
    return filepath, s

def interpret(series, name):
    s = series.sort_values(ascending=False)
    total = s.sum()
    top = s.idxmax()
    msg = [f"Interpretation for {name}:",
           f"Total = {total:.2f}. Top category is '{top}' contributing {(s[top]/total)*100:.2f}% of total.",
           "Detailed distribution:"]
    for k, v in s.items():
        msg.append(f" - {k}: {v:.2f} ({(v/total)*100:.2f}%)")
    return "\n".join(msg)

plots_info = []
texts = []

# 1. Product category pie
if product_col:
    prod_sum = df.groupby(product_col)[sales_col].sum()
    path, s = plot_pie(prod_sum, f"Sales by {product_col}", "sales_by_product.png")
    plots_info.append((path, f"Pie Chart: Sales by {product_col}"))
    texts.append(interpret(s, f"Sales by {product_col}"))

# 2. Gender pie
if gender_col:
    gen_sum = df.groupby(gender_col)[sales_col].sum()
    path, s = plot_pie(gen_sum, f"Sales by {gender_col}", "sales_by_gender.png")
    plots_info.append((path, f"Pie Chart: Sales by {gender_col}"))
    texts.append(interpret(s, f"Sales by {gender_col}"))

# 3. AgeGroup pie
if "AgeGroup" in df.columns:
    age_sum = df.groupby("AgeGroup")[sales_col].sum()
    path, s = plot_pie(age_sum, "Sales by Age Group", "sales_by_agegroup.png")
    plots_info.append((path, "Pie Chart: Sales by Age Group"))
    texts.append(interpret(s, "Sales by Age Group"))

# 4. Top customers pie
if id_col:
    cust_sum = df.groupby(id_col)[sales_col].sum().sort_values(ascending=False)
    topn = 8
    top = cust_sum.head(topn)
    other = cust_sum.iloc[topn:].sum()
    top["Other"] = other
    path, s = plot_pie(top, "Top 8 Customers + Others", "top_customers.png")
    plots_info.append((path, "Pie Chart: Top 8 Customers + Others"))
    texts.append(interpret(s, "Top 8 Customers + Others"))

# 5. Cross: product × gender
if product_col and gender_col:
    combo = df.groupby([product_col, gender_col])[sales_col].sum().unstack(fill_value=0)
    for prod in combo.index:
        s = combo.loc[prod]
        if (s > 0).sum() > 1:
            filename = f"{prod}_by_gender.png".replace(" ", "_")
            path, _ = plot_pie(s, f"{gender_col} share for {prod}", filename)
            plots_info.append((path, f"Pie Chart: {gender_col} share for {prod}"))
            texts.append(interpret(s, f"{gender_col} share for {prod}"))

# 6. Cross: product × agegroup
if product_col and "AgeGroup" in df.columns:
    combo = df.groupby([product_col, "AgeGroup"])[sales_col].sum().unstack(fill_value=0)
    for prod in combo.index:
        s = combo.loc[prod]
        if (s > 0).sum() > 1:
            filename = f"{prod}_by_agegroup.png".replace(" ", "_")
            path, _ = plot_pie(s, f"Age Group share for {prod}", filename)
            plots_info.append((path, f"Pie Chart: Age Group share for {prod}"))
            texts.append(interpret(s, f"Age Group share for {prod}"))

# ==========================
# GENERATE PDF REPORT
# ==========================
styles = getSampleStyleSheet()
report = SimpleDocTemplate(PDF_FILE, pagesize=A4)
story = []

title = Paragraph("Pie Chart Interpretation Report", styles["Title"])

story.append(title)
story.append(Spacer(1, 0.3*inch))

for (path, caption), text in zip(plots_info, texts):
    story.append(Paragraph(f"<b>{caption}</b>", styles["Heading2"]))
    story.append(Image(path, width=4.5*inch, height=4.5*inch))
    story.append(Spacer(1, 0.2*inch))
    story.append(Paragraph(text.replace("\n", "<br/>"), styles["BodyText"]))
    story.append(PageBreak())

report.build(story)
print(f"PDF Report {PDF_FILE}")

C:\Users\Dell\AppData\Local\Temp\ipykernel_38436\3388769023.py:93: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_sum = df.groupby("AgeGroup")[sales_col].sum()
C:\Users\Dell\AppData\Local\Temp\ipykernel_38436\3388769023.py:122: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  combo = df.groupby([product_col, "AgeGroup"])[sales_col].sum().unstack(fill_value=0)


PDF Report Retail_Sales_Pie_Chart_Report.pdf
